In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, zipfile, shutil, platform, html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
try:
    from IPython.display import display, FileLink
except ImportError:
    display = print
    FileLink = str

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/lead_hand_path_shape'
FIG = ROOT / 'figures/lead_hand_path_shape'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
LOAD_OPTIONAL_OUTCOMES = True
MIN_SWINGS = 5
PRIMARY_CUTOFF_HZ = 12.0
SELECTED_PROFILE = None
SEED = 20261020
KEYS = ['hitter_id', 'session_id', 'hitter_side']
METRICS = ['net_drop_cm', 'max_drop_cm', 'arc_length_cm', 'horizontal_sweep_deg',
           'approach_angle_deg', 'mean_curvature_per_m']
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
WINDOWS = {'plant': None, 'late100': .100, 'late75': .075, 'late125': .125}
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
ARCHIVE_URL = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/hitting_landmarks.zip'
ARCHIVE_HASH = '9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3'
if MIN_SWINGS < 3 or PRIMARY_CUTOFF_HZ <= 0:
    raise ValueError('Use at least three swings and a positive filter cutoff.')
plt.style.use('seaborn-v0_8-whitegrid')
print('Study 17: Lead-Hand Path Shape Through Rotation')
print('Operationalizes the supplied “down and around” observation using the lead wrist joint center.')
print('X: toward pitcher; Y: right-handed box; Z: upward. Left-handed Y is mirrored for display and sweep.')
print('This is a hand-path proxy, not barrel path, bat attack angle, or a mechanics quality grade.')

In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, target):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {target}; enable AUTO_DOWNLOAD or supply official inputs.')
    print('Downloading', target.name, flush=True)
    tmp = target.with_suffix(target.suffix + '.part')
    with urlopen(url, timeout=120) as source, tmp.open('wb') as dest:
        shutil.copyfileobj(source, dest)
    tmp.replace(target)

if not (RAW / 'landmarks.csv').exists():
    archive = RAW / 'hitting_landmarks.zip'
    if not archive.exists():
        download(ARCHIVE_URL, archive)
    if file_hash(archive) != ARCHIVE_HASH:
        raise ValueError('Landmark archive SHA-256 mismatch.')
    with zipfile.ZipFile(archive) as z:
        names = [n for n in z.namelist() if Path(n).name == 'landmarks.csv']
        if len(names) != 1:
            raise ValueError('Expected exactly one landmarks.csv.')
        target = RAW / 'landmarks.csv'
        tmp = target.with_suffix('.csv.part')
        with z.open(names[0]) as source, tmp.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        tmp.replace(target)
if not (RAW / 'metadata.csv').exists():
    download(CSV_BASE + 'metadata.csv', RAW / 'metadata.csv')
BASES = ['lwjc', 'rwjc', 'lhjc', 'rhjc', 'left_hip', 'right_hip']
COORDS = [f'{b}_{a}' for b in BASES for a in 'xyz']
landmarks = pd.read_csv(RAW / 'landmarks.csv', usecols=['session_swing', 'time', *EVENTS, *COORDS],
                        dtype={'session_swing': str})
metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
if metadata.session_swing.isna().any() or metadata.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate metadata swing key.')
if landmarks.session_swing.isna().any() or landmarks.duplicated(['session_swing', 'time']).any():
    raise ValueError('Missing landmark swing key or duplicate swing/time key.')
if metadata[['user', 'session', 'hitter_side']].isna().any().any():
    raise ValueError('Missing athlete/session/side identity.')
meta = metadata.set_index('session_swing')
groups = {str(k): g.sort_values('time').reset_index(drop=True)
          for k, g in landmarks.groupby('session_swing', sort=False)}
poi = pd.DataFrame(columns=['session_swing', 'bat_speed_mph_contact_x', 'exit_velo_mph_x'])
optional_status = 'disabled'
if LOAD_OPTIONAL_OUTCOMES:
    try:
        if not (RAW / 'poi_metrics.csv').exists():
            download(CSV_BASE + 'poi/poi_metrics.csv', RAW / 'poi_metrics.csv')
        poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
        optional_status = 'loaded'
    except OSError as exc:
        optional_status = str(exc)
        print('Optional outcomes unavailable:', exc)
if poi.session_swing.isna().any() or poi.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate POI swing key.')
for col in ['bat_speed_mph_contact_x', 'exit_velo_mph_x']:
    if col not in poi:
        poi[col] = np.nan
print(f'{len(groups)} landmark trials; {len(metadata)} metadata trials.')

In [ ]:
def get_events(g):
    ev = {}
    for name in EVENTS:
        values = g[name].to_numpy(float)
        valid = np.isfinite(values).all() and np.ptp(values) <= 1e-7
        if name == 'contact_time' and not valid:
            raise ValueError('missing_or_inconsistent_contact_time')
        ev[name] = float(values[0]) if valid else np.nan
    return ev

def sample_path(t, p, q):
    return np.column_stack([np.interp(q, t, p[:, a]) for a in range(3)])

def path_metrics(t, p, sweep=False):
    if len(t) < 8 or not np.isfinite(p).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid_metric_input')
    step = np.linalg.norm(np.diff(p, axis=0), axis=1)
    velocity = np.gradient(p, t, axis=0, edge_order=2)
    acceleration = np.gradient(velocity, t, axis=0, edge_order=2)
    speed = np.linalg.norm(velocity, axis=1)
    valid = (speed > .1)
    valid[:2] = False
    valid[-2:] = False
    curvature = np.full(len(t), np.nan)
    curvature[valid] = np.linalg.norm(np.cross(velocity[valid], acceleration[valid]), axis=1) / speed[valid]**3
    weights = speed[valid]
    average_curvature = float(np.average(curvature[valid], weights=weights)) if len(weights) else np.nan
    q = np.linspace(t[-1] - .020, t[-1], 9)
    if q[0] < t[0] - 1e-9:
        raise ValueError('window_shorter_than_20ms')
    approach = sample_path(t, p, q)
    centered = q - q.mean()
    v = centered @ approach / (centered @ centered)
    horizontal_speed = float(np.linalg.norm(v[:2]))
    approach_angle = float(np.degrees(np.arctan2(v[2], horizontal_speed))) if np.linalg.norm(v) > .1 else np.nan
    radius = np.linalg.norm(p[:, :2], axis=1)
    angle = np.unwrap(np.arctan2(p[:, 1], p[:, 0]))
    angular_sweep = float(np.degrees(angle[-1] - angle[0])) if sweep and radius.min() >= .05 else np.nan
    minimum = int(np.argmin(p[:, 2]))
    return dict(net_drop_cm=float(100 * (p[0, 2] - p[-1, 2])),
                max_drop_cm=float(100 * (p[0, 2] - p[minimum, 2])),
                arc_length_cm=float(100 * step.sum()),
                horizontal_sweep_deg=angular_sweep,
                approach_angle_deg=approach_angle,
                mean_curvature_per_m=average_curvature,
                lowest_height_time_ms=float(1000 * (t[minimum] - t[-1])),
                min_horizontal_radius_cm=float(100 * radius.min()) if sweep else np.nan,
                curvature_valid_fraction=float(valid.mean()))

def prepare_trial(g, side, cutoff=12., landmark='wrist'):
    if side not in ('R', 'L'):
        raise ValueError('invalid_hitter_side')
    ev = get_events(g)
    t = g.time.to_numpy(float)
    if len(t) < 30 or not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid_time_grid')
    dt = float(np.median(np.diff(t)))
    if not np.allclose(np.diff(t), dt, atol=.00011, rtol=.03):
        raise ValueError('irregular_sampling_or_gap')
    contact = ev['contact_time']
    if not t[0] <= contact <= t[-1]:
        raise ValueError('contact_outside_recording')
    end = min(len(t), np.searchsorted(t, contact, side='left') + 1)
    t = t[:end]
    prefix = ('l' if side == 'R' else 'r') + ('wjc' if landmark == 'wrist' else 'hjc')
    hand = g[[f'{prefix}_{a}' for a in 'xyz']].to_numpy(float)[:end]
    pelvis = (g[[f'left_hip_{a}' for a in 'xyz']].to_numpy(float)[:end]
              + g[[f'right_hip_{a}' for a in 'xyz']].to_numpy(float)[:end]) / 2
    if not np.isfinite(hand).all() or not np.isfinite(pelvis).all():
        raise ValueError('nonfinite_precontact_coordinates')
    if not .3 < float(np.median(pelvis[:, 2])) < 1.6:
        raise ValueError('pelvis_height_inconsistent_with_metre_units')
    uniform_t = np.linspace(t[0], contact, int(round((contact-t[0])/dt)) + 1)
    if len(uniform_t) < 30:
        raise ValueError('insufficient_precontact_filter_samples')
    h = sample_path(t, hand, uniform_t)
    pelvis = sample_path(t, pelvis, uniform_t)
    if cutoff is not None:
        fs = 1 / np.median(np.diff(uniform_t))
        if not 0 < cutoff < fs / 2:
            raise ValueError('invalid_filter_cutoff')
        sos = butter(4, cutoff, fs=fs, output='sos')
        h = sosfiltfilt(sos, h, axis=0)
        pelvis = sosfiltfilt(sos, pelvis, axis=0)
    signs = np.array([1., 1. if side == 'R' else -1., 1.])
    return uniform_t, {'global': h * signs, 'pelvis_relative': (h - pelvis) * signs}, ev

def extract_window(t, paths, ev, window):
    contact = ev['contact_time']
    if window == 'plant':
        if not ev['fp_10_time'] < ev['fp_100_time'] < contact:
            raise ValueError('unordered_foot_contact_plant_contact')
        start = ev['fp_100_time']
    else:
        start = contact - WINDOWS[window]
    dt = float(np.median(np.diff(t)))
    if start < t[0] - 1e-9 or contact - start < max(.020, 7 * dt):
        raise ValueError('incomplete_or_short_window')
    q = np.r_[start, t[(t > start + 1e-10) & (t < contact - 1e-10)], contact]
    measured, curves = {}, {}
    for frame, xyz in paths.items():
        p = sample_path(t, xyz, q)
        measured[frame] = path_metrics(q, p, sweep=frame == 'pelvis_relative')
        measured[frame]['duration_ms'] = float(1000 * (contact - start))
        grid = np.linspace(start, contact, 101)
        curves[frame] = sample_path(t, xyz, grid)
    return measured, curves

In [ ]:
def measurement_checks():
    t = np.linspace(0, .1, 101)
    p = np.column_stack([2*t, np.zeros(len(t)), 1 - t])
    m = path_metrics(t, p)
    assert np.isclose(m['net_drop_cm'], 10)
    assert np.isclose(m['arc_length_cm'], np.sqrt(.2**2 + .1**2)*100)
    assert np.isclose(m['approach_angle_deg'], np.degrees(np.arctan2(-1,2)))
    assert abs(m['mean_curvature_per_m']) < 1e-8
    theta = np.linspace(-.8, .8, 1001)
    t = np.linspace(0, .1, len(theta))
    circle = np.column_stack([.4*np.cos(theta), .4*np.sin(theta), np.zeros(len(t))])
    m = path_metrics(t, circle, True)
    assert np.isclose(m['horizontal_sweep_deg'], np.degrees(1.6))
    assert np.isclose(m['arc_length_cm'], 64, atol=.001)
    assert np.isclose(m['mean_curvature_per_m'], 2.5, rtol=.001)
    theta = np.deg2rad(np.linspace(170,210,101))
    wrapped = np.column_stack([np.cos(theta), np.sin(theta), np.zeros(101)])
    assert np.isclose(path_metrics(np.linspace(0,.1,101),wrapped,True)['horizontal_sweep_deg'],40)
    rise = p.copy()
    rise[:,2] = 1 + np.abs(np.linspace(-.05,.05,101))
    m = path_metrics(np.linspace(0,.1,101),rise)
    assert abs(m['net_drop_cm']) < 1e-10 and np.isclose(m['max_drop_cm'],5)
    moving_pelvis = np.column_stack([t*2,t*3,t])
    fixed_offset = np.tile([.2,.3,.4],(len(t),1))
    relative = moving_pelvis + fixed_offset - moving_pelvis
    assert path_metrics(t, relative)['arc_length_cm'] < 1e-10
    assert np.isnan(path_metrics(t, relative)['approach_angle_deg'])
    print('Analytical measurement checks passed: line, circular arc, angle wrap, drop/rebound, translation.')

measurement_checks()
CONFIGS = [('primary', PRIMARY_CUTOFF_HZ, 'wrist', ['plant','late100']),
           ('raw', None, 'wrist', ['plant','late100']),
           ('filter8', 8., 'wrist', ['plant','late100']),
           ('filter16', 16., 'wrist', ['plant','late100']),
           ('window75', PRIMARY_CUTOFF_HZ, 'wrist', ['late75']),
           ('window125', PRIMARY_CUTOFF_HZ, 'wrist', ['late125']),
           ('hand_center', PRIMARY_CUTOFF_HZ, 'hand', ['plant','late100'])]
rows, audit, curves = [], [], {}
for swing, g in groups.items():
    if swing not in meta.index:
        audit.append(dict(session_swing=swing, config='all', window='all', status='missing_metadata'))
        continue
    r = meta.loc[swing]
    identity = dict(session_swing=swing, hitter_id=r.user, session_id=r.session, hitter_side=r.hitter_side)
    height = float(r.session_height_in) * .0254
    for config, cutoff, marker, windows in CONFIGS:
        try:
            t, paths, ev = prepare_trial(g, r.hitter_side, cutoff, marker)
        except ValueError as exc:
            for window in windows:
                audit.append({**identity, 'config':config, 'window':window, 'status':str(exc)})
            continue
        for window in windows:
            try:
                features, trajectory = extract_window(t, paths, ev, window)
                for frame, values in features.items():
                    row = {**identity, 'config':config, 'window':window, 'frame':frame,
                           'height_m':height, **values}
                    for metric in ('net_drop_cm','max_drop_cm','arc_length_cm'):
                        row[metric.replace('_cm','_pct_height')] = row[metric] / height if np.isfinite(height) and height > 0 else np.nan
                    rows.append(row)
                    if config == 'primary':
                        curves[(swing, window, frame)] = trajectory[frame]
                audit.append({**identity, 'config':config, 'window':window, 'status':'retained'})
            except ValueError as exc:
                audit.append({**identity, 'config':config, 'window':window, 'status':str(exc)})
measurements = pd.DataFrame(rows)
audit = pd.DataFrame(audit)
if measurements.empty:
    raise ValueError('No eligible trials; inspect input conventions and audit.')
measurements = measurements.merge(poi[['session_swing','bat_speed_mph_contact_x','exit_velo_mph_x']],
                                  on='session_swing', how='left', validate='many_to_one')
measurements.to_csv(OUT / 'swing_measurements.csv', index=False)
audit.to_csv(OUT / 'eligibility_audit.csv', index=False)
primary = measurements[measurements.config.eq('primary')].copy()
print(audit.groupby(['config','window','status']).size().to_string())

In [ ]:
def make_profiles(table, minimum):
    group_keys = ['config', 'window', 'frame', *KEYS]
    grouped = table.groupby(group_keys, dropna=False)
    features = [*METRICS, 'net_drop_pct_height','arc_length_pct_height', 'duration_ms',
                'bat_speed_mph_contact_x','exit_velo_mph_x']
    records = []
    for key, g in grouped:
        if len(g) < minimum:
            continue
        record = dict(zip(group_keys,key))
        record['n_swings'] = len(g)
        for metric in features:
            values = g[metric].to_numpy(float)
            values = values[np.isfinite(values)]
            record[metric + '_mean'] = float(values.mean()) if len(values) else np.nan
            record[metric + '_sd'] = float(values.std(ddof=1)) if len(values)>1 else np.nan
        if key[0] == 'primary':
            cube = np.stack([curves[(s,key[1],key[2])] for s in g.session_swing])
            spread = cube.std(axis=0,ddof=1)
            record['trajectory_rms_sd_cm'] = float(100*np.sqrt(np.mean(np.sum(spread**2,axis=1))))
            relative = cube - cube[:, :1, :]
            record['start_relative_trajectory_rms_sd_cm'] = float(100*np.sqrt(np.mean(np.sum(relative.std(axis=0,ddof=1)**2,axis=1))))
        records.append(record)
    return pd.DataFrame(records)

profiles = make_profiles(measurements, MIN_SWINGS)
profiles.to_csv(OUT / 'session_profiles.csv', index=False)
minimum_audit = []
for minimum in (4,5,6):
    p = make_profiles(primary, minimum)
    for (window,frame), g in p.groupby(['window','frame']):
        minimum_audit.append(dict(minimum=minimum,window=window,frame=frame,profiles=len(g),
                                  hitters=g.hitter_id.nunique(),median_trajectory_rms_sd_cm=g.trajectory_rms_sd_cm.median()))
pd.DataFrame(minimum_audit).to_csv(OUT / 'minimum_swing_sensitivity.csv', index=False)
comparisons = []
base = measurements[measurements.config.eq('primary')]
match_keys = ['session_swing','window','frame']
for config in measurements.config.unique():
    if config == 'primary':
        continue
    alt = measurements[measurements.config.eq(config)].copy()
    if config.startswith('window'):
        alt['window'] = 'late100'
    matched = base.merge(alt, on=match_keys, suffixes=('_base','_alt'), validate='one_to_one')
    for (window, frame), g in matched.groupby(['window','frame']):
        for metric in METRICS:
            a,b = g[metric+'_base'].to_numpy(float),g[metric+'_alt'].to_numpy(float)
            valid = np.isfinite(a) & np.isfinite(b)
            a,b=a[valid],b[valid]
            comparisons.append(dict(config=config,window=window,frame=frame,metric=metric,n=len(a),
                                    median_change=float(np.median(b-a)) if len(a) else np.nan,
                                    median_absolute_change=float(np.median(np.abs(b-a))) if len(a) else np.nan,
                                    rank_correlation=float(spearmanr(a,b).statistic) if len(a)>2 and np.ptp(a)>0 and np.ptp(b)>0 else np.nan))
sensitivity = pd.DataFrame(comparisons)
sensitivity.to_csv(OUT / 'matched_sensitivity.csv', index=False)
duration = primary[primary.frame.eq('global')].copy()
duration['plant_outside_80_200ms'] = duration.window.eq('plant') & ~duration.duration_ms.between(80,200)
duration.to_csv(OUT / 'duration_review.csv', index=False)
screened = primary[~(primary.window.eq('plant') & ~primary.duration_ms.between(80,200))]
screened_profiles = make_profiles(screened,MIN_SWINGS)
screened_profiles.to_csv(OUT / 'duration_screened_profiles.csv', index=False)
summary = []
rng = np.random.default_rng(SEED)
for (window,frame), g in profiles[profiles.config.eq('primary')].groupby(['window','frame']):
    for metric in METRICS:
        values = g.groupby('hitter_id')[metric+'_mean'].mean().dropna().to_numpy(float)
        if len(values):
            samples = rng.choice(values,(1000,len(values)),replace=True).mean(axis=1)
            summary.append(dict(window=window,frame=frame,metric=metric,hitters=len(values),
                                hitter_weighted_mean=float(values.mean()),
                                bootstrap_low=float(np.quantile(samples,.025)),
                                bootstrap_high=float(np.quantile(samples,.975))))
pd.DataFrame(summary).to_csv(OUT / 'hitter_weighted_summary.csv', index=False)
print('Primary session profiles:')
display(profiles[profiles.config.eq('primary')][['window','frame',*KEYS,'n_swings','net_drop_cm_mean','arc_length_cm_mean','horizontal_sweep_deg_mean']].head(12))

SENSITIVITY_FRAME = 'pelvis_relative'
if SENSITIVITY_FRAME not in ('global', 'pelvis_relative'):
    raise ValueError('SENSITIVITY_FRAME must be global or pelvis_relative.')
SENSITIVITY_LABELS = {
    'raw': 'No additional filter',
    'filter8': '8 Hz filter',
    'filter16': '16 Hz filter',
    'window75': 'Final 75 ms',
    'window125': 'Final 125 ms',
    'hand_center': 'Hand joint center',
}
SENSITIVITY_METRICS = {
    'net_drop_cm': 'Net drop (cm)',
    'max_drop_cm': 'Max drop (cm)',
    'arc_length_cm': 'Arc length (cm)',
    'horizontal_sweep_deg': 'Sweep (deg)',
    'approach_angle_deg': 'Approach angle (deg)',
    'mean_curvature_per_m': 'Curvature (1/m)',
}
print('\nSensitivity review: ' + SENSITIVITY_FRAME.replace('_', ' '))
print('Reference: lead wrist, additional 12 Hz filter, matching primary window.')
print('Changes are alternative minus reference on matched swings; positive net-drop change means more downward movement.')
print('Window alternatives compare 75/125 ms with 100 ms, so differences also reflect the amount of swing observed.')
print('Rank correlation describes ordering; it does not establish agreement in absolute measurements.')
if SENSITIVITY_FRAME == 'global':
    print('Horizontal sweep is unavailable in global coordinates by design.')
review = sensitivity[sensitivity.frame.eq(SENSITIVITY_FRAME)].copy()
review.to_csv(OUT / 'visible_sensitivity_summary.csv', index=False)
counts = review.groupby(['window', 'config']).agg(
    minimum_metric_n=('n', 'min'), maximum_metric_n=('n', 'max')).reset_index()
counts['alternative'] = counts.config.map(SENSITIVITY_LABELS)
print('\nMatched swing counts; metric-specific counts can differ when a measurement is unavailable:')
display(counts[['window', 'alternative', 'minimum_metric_n', 'maximum_metric_n']])
for window in ('plant', 'late100'):
    part = review[review.window.eq(window)].copy()
    if part.empty:
        continue
    order = [c for c in SENSITIVITY_LABELS if c in part.config.unique()]
    print('\n' + ('Foot plant to contact' if window == 'plant' else 'Final 100 ms') + ' sensitivity')
    for value, title in [('median_change', 'Median signed change: alternative minus reference'),
                         ('median_absolute_change', 'Median absolute difference from reference'),
                         ('rank_correlation', 'Spearman rank correlation with reference')]:
        table = part.pivot(index='config', columns='metric', values=value)
        table = table.reindex(index=order, columns=list(SENSITIVITY_METRICS))
        table.index = table.index.map(SENSITIVITY_LABELS)
        table = table.rename(columns=SENSITIVITY_METRICS)
        table.index.name = 'Alternative'
        print(title)
        display(table.round(3))
        table.to_csv(OUT / f'sensitivity_{window}_{value}_{SENSITIVITY_FRAME}.csv')
fig, axes = plt.subplots(2, 3, figsize=(16, 10), squeeze=False)
for row_index, window in enumerate(('plant', 'late100')):
    for col_index, metric in enumerate(('net_drop_cm', 'arc_length_cm', 'approach_angle_deg')):
        ax = axes[row_index, col_index]
        part = review[review.window.eq(window) & review.metric.eq(metric)].set_index('config')
        order = [c for c in SENSITIVITY_LABELS if c in part.index and np.isfinite(part.loc[c, 'median_change'])]
        if not order:
            ax.text(.5, .5, 'No matched measurements', ha='center', va='center', transform=ax.transAxes)
            continue
        values = part.loc[order, 'median_change'].to_numpy(float)
        positions = np.arange(len(order))
        colors = ['#327b91' if c in ('raw', 'filter8', 'filter16') else '#a1678c' if c.startswith('window') else '#b17b35' for c in order]
        ax.barh(positions, values, color=colors)
        ax.set_yticks(positions, [SENSITIVITY_LABELS[c] for c in order])
        ax.invert_yaxis()
        ax.axvline(0, color='black', linewidth=.8)
        ax.set(title=('Plant to contact' if window == 'plant' else 'Final 100 ms') + ' | ' + SENSITIVITY_METRICS[metric],
               xlabel='Median change from 12 Hz lead-wrist reference')
        low, high = min(0., float(values.min())), max(0., float(values.max()))
        padding = max((high - low) * .22, .05)
        ax.set_xlim(low - padding, high + padding)
        for position, value in zip(positions, values):
            ax.annotate(f'{value:+.2f}', (value, position), xytext=(4 if value >= 0 else -4, 0),
                        textcoords='offset points', va='center', ha='left' if value >= 0 else 'right', fontsize=9)
fig.suptitle('Processing sensitivity | ' + SENSITIVITY_FRAME.replace('_', ' ') + '\nEach bar uses exactly matched swings; units are shown in panel titles', fontsize=14)
fig.tight_layout(rect=(0, 0, 1, .95))
fig.savefig(FIG / 'sensitivity_comparisons.png', dpi=160, bbox_inches='tight')
plt.show()
focus = review[review.window.eq('late100')].set_index(['config', 'metric'])
print('\nFinal-100-ms interpretation checks:')
for config in ('raw', 'filter8', 'filter16', 'hand_center'):
    for metric in ('net_drop_cm', 'arc_length_cm', 'approach_angle_deg'):
        if (config, metric) in focus.index:
            r = focus.loc[(config, metric)]
            print(f"{SENSITIVITY_LABELS[config]} | {SENSITIVITY_METRICS[metric]}: signed {r.median_change:+.2f}; absolute {r.median_absolute_change:.2f}; rank rho {r.rank_correlation:.3f}; n={int(r.n)}")
print('Landmark and filter differences are measurement sensitivity, not evidence of better or worse mechanics.')


In [ ]:
associations = []
for window in ('plant','late100'):
    p = profiles[(profiles.config.eq('primary')) & profiles.window.eq(window) & profiles.frame.eq('pelvis_relative')]
    athlete = p.groupby('hitter_id')[[m+'_mean' for m in METRICS]+['bat_speed_mph_contact_x_mean','exit_velo_mph_x_mean']].mean()
    for metric in ['net_drop_cm','arc_length_cm','horizontal_sweep_deg']:
        for outcome in ['bat_speed_mph_contact_x','exit_velo_mph_x']:
            a = athlete[[metric+'_mean',outcome+'_mean']].dropna()
            if len(a)<10 or (a.nunique()<2).any():
                continue
            result = spearmanr(a.iloc[:,0],a.iloc[:,1])
            associations.append(dict(window=window,frame='pelvis_relative',metric=metric,outcome=outcome,
                                     hitters=len(a),rho=float(result.statistic),p_value=float(result.pvalue)))
associations = pd.DataFrame(associations,columns=['window','frame','metric','outcome','hitters','rho','p_value'])
if len(associations):
    order = np.argsort(associations.p_value.to_numpy())
    adjusted = associations.p_value.to_numpy()[order] * len(order) / np.arange(1,len(order)+1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1].clip(0,1)
    associations.loc[order,'fdr_q_value'] = adjusted
associations.to_csv(OUT / 'exploratory_outcome_associations.csv', index=False)
print('Exploratory between-hitter associations: equal weight per hitter; FDR across all reported tests.')
print('No adjustment for pitch location, contact height, physical traits, or playing level; no causal inference.')
display(associations)

In [ ]:
profile_data = profiles[(profiles.config.eq('primary')) & profiles.window.eq('late100') & profiles.frame.eq('pelvis_relative')]
if profile_data.empty:
    raise ValueError('No late100 profiles meet the swing minimum; lower MIN_SWINGS deliberately or inspect audit.')
profile_keys = list(profile_data[KEYS].itertuples(index=False,name=None))
selected_key = tuple(SELECTED_PROFILE) if SELECTED_PROFILE is not None else sorted(profile_keys)[0]
if selected_key not in profile_keys:
    raise ValueError('SELECTED_PROFILE must match an eligible hitter/session/side.')
selected = primary[primary.window.eq('late100') & primary.frame.eq('pelvis_relative')]
selected = selected[(selected[KEYS] == pd.Series(selected_key,index=KEYS)).all(axis=1)]
ids = selected.session_swing.tolist()
progress_ms = np.linspace(-100,0,101)
fig, axes = plt.subplots(2,3,figsize=(16,9))
for col, frame in enumerate(('global','pelvis_relative')):
    cube = np.stack([curves[(s,'late100',frame)] for s in ids])*100
    start = cube - cube[:,:1,:]
    for p in start:
        axes[0,col].plot(p[:,0],p[:,2],alpha=.55,lw=1.2)
        axes[0,col].scatter(p[-1,0],p[-1,2],s=15)
        axes[1,col].plot(progress_ms,p[:,2],alpha=.5,lw=1)
    mean,sd = start[:,:,2].mean(axis=0), start[:,:,2].std(axis=0,ddof=1)
    axes[1,col].plot(progress_ms,mean,color='black',lw=2.3)
    axes[1,col].fill_between(progress_ms,mean-sd,mean+sd,color='black',alpha=.12)
    axes[0,col].set(title=frame.replace('_',' ').title(),xlabel='Forward displacement from window start (cm)',ylabel='Height change from window start (cm)')
    axes[0,col].set_aspect('equal',adjustable='datalim')
    axes[1,col].set(xlabel='Time to contact (ms)',ylabel='Height change (cm)')
cube = np.stack([curves[(s,'late100','pelvis_relative')] for s in ids])*100
for p in cube:
    axes[0,2].plot(p[:,0],p[:,1],alpha=.6,lw=1.2)
    axes[0,2].scatter(p[-1,0],p[-1,1],s=15)
axes[0,2].scatter([0],[0],marker='+',s=100,color='black',label='Hip midpoint')
axes[0,2].set(title='Around the pelvis: overhead',xlabel='Forward relative to hips (cm)',ylabel='Lateral relative to hips (cm; L mirrored)')
axes[0,2].set_aspect('equal',adjustable='datalim')
p = cube[0]
segments = np.stack([p[:-1,[0,2]],p[1:,[0,2]]],axis=1)
lc = LineCollection(segments,cmap='viridis',norm=plt.Normalize(-100,0),linewidth=3)
lc.set_array(progress_ms[:-1]);axes[1,2].add_collection(lc);axes[1,2].autoscale()
axes[1,2].scatter(p[-1,0],p[-1,2],color='black',s=30)
axes[1,2].set(title=f'Example swing {ids[0]}',xlabel='Forward relative to hips (cm)',ylabel='Height relative to hips (cm)')
fig.colorbar(lc,ax=axes[1,2],label='Time to contact (ms)')
fig.suptitle(f'Lead-wrist path | hitter {selected_key[0]} | session {selected_key[1]} | {selected_key[2]} | {len(ids)} swings')
fig.tight_layout()
fig.savefig(FIG / 'selected_hitter_profile.png',dpi=160,bbox_inches='tight')
plt.show()
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax, metric, label in zip(axes,['net_drop_cm_mean','arc_length_cm_mean','horizontal_sweep_deg_mean'],
                              ['Net downward movement (cm)','3D arc length (cm)','Signed horizontal sweep (degrees)']):
    values=profile_data.groupby('hitter_id')[metric].mean().dropna()
    ax.hist(values,bins=18,color='#327b91',edgecolor='white')
    ax.set(xlabel=label,ylabel='Hitters',title='Pelvis-relative · final 100 ms')
fig.tight_layout();fig.savefig(FIG / 'hitter_distributions.png',dpi=160,bbox_inches='tight');plt.show()

In [ ]:
report_profiles=[]
for key in sorted(profile_keys):
    mask=(primary[KEYS] == pd.Series(key,index=KEYS)).all(axis=1) & primary.window.eq('late100')
    ids=primary[mask].session_swing.unique().tolist()
    item={'key':list(key),'ids':ids,'curves':{},'metrics':{}}
    for frame in ('global','pelvis_relative'):
        item['curves'][frame]=[np.round(curves[(s,'late100',frame)]*100,3).tolist() for s in ids]
        p=profiles[(profiles.config.eq('primary')) & profiles.window.eq('late100') & profiles.frame.eq(frame)]
        row=p[(p[KEYS] == pd.Series(key,index=KEYS)).all(axis=1)].iloc[0]
        item['metrics'][frame]={m:float(row[m+'_mean']) if pd.notna(row[m+'_mean']) else None for m in METRICS}
    report_profiles.append(item)
REPORT = r"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Lead-Hand Path Profiles</title><style>
body{font:15px system-ui,sans-serif;background:#f3f6f8;color:#172937;max-width:1450px;margin:auto;padding:24px}h1{margin-bottom:8px}p{line-height:1.6;color:#475d6b}.controls,.card{background:white;padding:18px;border:1px solid #d6e0e7;border-radius:8px;margin:14px 0}.controls{display:flex;gap:18px;flex-wrap:wrap}select{padding:8px;max-width:95%;border:1px solid #a5bac7;border-radius:4px}.grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px}.card{min-width:0}canvas{width:100%;height:auto}h2{font-size:17px}table{width:100%;border-collapse:collapse}td,th{text-align:left;padding:9px;border-bottom:1px solid #d6e0e7}.note{padding:14px;border-left:4px solid #327b91;background:#e6f0f4}@media(max-width:950px){.grid{grid-template-columns:1fr}}
</style></head><body><h1>Lead-Hand Path Shape</h1><p>Study 17 · OpenBiomechanics Dataset v1 · Final 100 ms before contact</p><p class="note">“Down and around” is represented by vertical movement and horizontal sweep around the hip midpoint. The lead wrist is a hand-path proxy. These measurements do not describe barrel path, bat attack angle, or an ideal swing.</p>
<div class="controls"><label>Hitter / session / batting side <select id="profile"></select></label><label>Coordinates <select id="frame"><option value="pelvis_relative">Relative to moving hip midpoint</option><option value="global">Global laboratory coordinates</option></select></label><label>Highlight swing <select id="swing"></select></label></div>
<p id="count"></p><div class="grid"><div class="card"><h2>Forward displacement versus height</h2><canvas id="depth"></canvas><p>Window-start-relative positions. Contact is marked with a dot; highlighted swing is colored by time from −100 ms (blue) to contact (gold).</p></div><div class="card"><h2>Overhead path</h2><canvas id="overhead"></canvas><p>Pelvis-relative coordinates retain distance from the hip midpoint (+). Global view uses window-start-relative positions. Left-handed Y is mirrored.</p></div><div class="card"><h2>Vertical movement through contact</h2><canvas id="height"></canvas><p>Each curve starts at zero height change. Black line: mean. Shading: ±1 across-swing sample SD, not a confidence interval.</p></div></div>
<div class="card"><h2>Session averages</h2><table><thead><tr><th>Measurement</th><th>Value</th></tr></thead><tbody id="metrics"></tbody></table></div>
<p>Positive drop means downward. Approach angle is upward (+) or downward (−), estimated from a linear fit over the last 20 ms. Horizontal sweep is signed counterclockwise after left-handed Y mirroring; available only relative to the pelvis. Pelvis subtraction removes translation but retains laboratory orientation. Additional 12 Hz smoothing uses the precontact recording with reflection at contact. Curvature and terminal direction are sensitive to processing choices; review the exported sensitivities.</p>
<script>
const data=__DATA__;
const $=id=>document.getElementById(id);const colors=['#327b91','#a1678c','#5c8766','#b17b35','#536fa2'];
data.forEach((p,i)=>$('profile').add(new Option(p.key.join(' / '),i)));
function line(ctx,pts,fx,fy,color,width=1){ctx.beginPath();pts.forEach((p,i)=>i?ctx.lineTo(fx(p[0]),fy(p[1])):ctx.moveTo(fx(p[0]),fy(p[1])));ctx.strokeStyle=color;ctx.lineWidth=width;ctx.stroke()}
function plot(id,series,xlabel,ylabel,highlight,{mean=false,origin=false,equal=false}={}){
 const cv=$(id);cv.width=700;cv.height=530;const ctx=cv.getContext('2d');ctx.clearRect(0,0,700,530);
 let points=series.flat(),xs=points.map(p=>p[0]),ys=points.map(p=>p[1]),loX=Math.min(...xs),hiX=Math.max(...xs),loY=Math.min(...ys),hiY=Math.max(...ys);
 let mu=[],sd=[];if(mean){mu=series[0].map((_,i)=>series.reduce((a,s)=>a+s[i][1],0)/series.length);sd=mu.map((m,i)=>Math.sqrt(series.reduce((a,s)=>a+(s[i][1]-m)**2,0)/(series.length-1)));loY=Math.min(loY,...mu.map((v,i)=>v-sd[i]));hiY=Math.max(hiY,...mu.map((v,i)=>v+sd[i]))}
 if(origin){loX=Math.min(loX,0);hiX=Math.max(hiX,0);loY=Math.min(loY,0);hiY=Math.max(hiY,0)}
 let px=Math.max((hiX-loX)*.12,1),py=Math.max((hiY-loY)*.12,1);loX-=px;hiX+=px;loY-=py;hiY+=py;
 const left=82,right=675,top=28,bottom=450;if(equal){const scale=Math.max((hiX-loX)/(right-left),(hiY-loY)/(bottom-top));let midX=(loX+hiX)/2,midY=(loY+hiY)/2;loX=midX-scale*(right-left)/2;hiX=midX+scale*(right-left)/2;loY=midY-scale*(bottom-top)/2;hiY=midY+scale*(bottom-top)/2}
 const fx=x=>left+(x-loX)/(hiX-loX)*(right-left),fy=y=>bottom-(y-loY)/(hiY-loY)*(bottom-top);
 ctx.font='17px system-ui';for(let i=0;i<=4;i++){let x=loX+(hiX-loX)*i/4,y=loY+(hiY-loY)*i/4;ctx.strokeStyle='#e0e8ed';ctx.beginPath();ctx.moveTo(left,fy(y));ctx.lineTo(right,fy(y));ctx.stroke();ctx.fillStyle='#475d6b';ctx.fillText(y.toFixed(1),8,fy(y)+5);ctx.fillText(x.toFixed(0),fx(x)-12,bottom+28)}
 if(mean){ctx.beginPath();mu.forEach((v,i)=>i?ctx.lineTo(fx(series[0][i][0]),fy(v+sd[i])):ctx.moveTo(fx(series[0][i][0]),fy(v+sd[i])));for(let i=100;i>=0;i--)ctx.lineTo(fx(series[0][i][0]),fy(mu[i]-sd[i]));ctx.closePath();ctx.fillStyle='#17293720';ctx.fill()}
 series.forEach((s,i)=>line(ctx,s,fx,fy,highlight===i?'#bb8422':colors[i%colors.length]+'70',highlight===i?3:1.5));
 if(mean)line(ctx,mu.map((v,i)=>[series[0][i][0],v]),fx,fy,'#172937',3);
 if(highlight>=0 && id==='depth'){const s=series[highlight];for(let i=0;i<100;i++)line(ctx,s.slice(i,i+2),fx,fy,`hsl(${210-i*1.65} 70% 43%)`,4)}
 if(!mean)series.forEach((s,i)=>{const p=s.at(-1);ctx.beginPath();ctx.arc(fx(p[0]),fy(p[1]),i===highlight?5:3,0,2*Math.PI);ctx.fillStyle=i===highlight?'#172937':colors[i%colors.length];ctx.fill()});
 if(origin){ctx.fillStyle='#172937';ctx.fillText('+',fx(0)-6,fy(0)+6)}
 ctx.fillStyle='#172937';ctx.fillText(xlabel,120,510);ctx.save();ctx.translate(22,350);ctx.rotate(-Math.PI/2);ctx.fillText(ylabel,0,0);ctx.restore()
}
function render(reset=false){const p=data[Number($('profile').value)],frame=$('frame').value;if(reset){$('swing').replaceChildren(new Option('All swings',-1));p.ids.forEach((id,i)=>$('swing').add(new Option(id,i)))}const selected=Number($('swing').value),cube=p.curves[frame];
 const relative=cube.map(s=>s.map(v=>v.map((q,j)=>q-s[0][j])));
 plot('depth',relative.map(s=>s.map(p=>[p[0],p[2]])),'Forward displacement (cm)','Height change (cm)',selected,{equal:true});
 plot('overhead',(frame==='global'?relative:cube).map(s=>s.map(p=>[p[0],p[1]])),frame==='global'?'Forward displacement (cm)':'Forward relative to hips (cm)',frame==='global'?'Lateral displacement (cm)':'Lateral relative to hips (cm)',selected,{origin:frame!=='global',equal:true});
 plot('height',relative.map(s=>s.map((p,i)=>[-100+i,p[2]])),'Time to contact (ms)','Height change (cm)',selected,{mean:true});
 $('count').textContent=`${p.ids.length} retained swings · Hitter ${p.key[0]} · Session ${p.key[1]} · ${p.key[2]}`;
 const labels={net_drop_cm:'Net vertical drop (cm)',max_drop_cm:'Maximum downward excursion (cm)',arc_length_cm:'3D path arc length (cm)',horizontal_sweep_deg:'Signed horizontal sweep (°)',approach_angle_deg:'Hand-path approach angle (°)',mean_curvature_per_m:'Speed-weighted curvature (1/m)'};
 $('metrics').replaceChildren();Object.entries(labels).forEach(([key,label])=>{const tr=document.createElement('tr'),td=document.createElement('td'),v=document.createElement('td');td.textContent=label;v.textContent=p.metrics[frame][key]===null?'Unavailable':p.metrics[frame][key].toFixed(2);tr.append(td,v);$('metrics').append(tr)})
}
$('profile').addEventListener('change',()=>render(true));$('frame').addEventListener('change',()=>render());$('swing').addEventListener('change',()=>render());render(true);
</script></body></html>
"""
(OUT / 'lead_hand_profiles.html').write_text(REPORT.replace('__DATA__',json.dumps(report_profiles,allow_nan=False)),encoding='utf-8')
manifest={'study':17,'title':'Lead-Hand Path Shape Through Rotation','python':platform.python_version(),
          'numpy':np.__version__,'pandas':pd.__version__,'primary_cutoff_hz':PRIMARY_CUTOFF_HZ,
          'minimum_swings':MIN_SWINGS,'landmark':'lead wrist joint center',
          'units':'processed landmarks in metres; lengths exported in cm',
          'filter':'4th-order Butterworth SOS, forward/backward, full precontact signal; reflection at contact',
          'left_handed_transform':'mirror Y only after anatomical lead-wrist selection',
          'optional_outcomes':optional_status,'input_hashes':{p.name:file_hash(p) for p in
              [RAW/'landmarks.csv',RAW/'metadata.csv',RAW/'poi_metrics.csv'] if p.exists()},
          'primary_eligible':primary.groupby(['window','frame']).size().rename('n_swings').reset_index().to_dict('records'),
          'primary_windows':{'plant':'foot plant to contact with ordered foot-contact/plant/contact events', 'late100':'fixed final 100 ms, independent of foot event availability'},
          'approach_fit_ms':20,'curvature_speed_floor_m_s':.1,'sweep_radius_floor_m':.05,
          'interpretation':'Hand trajectory proxy. No barrel path, ideal mechanics, coaching prescription, or causal effect.'}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Created',OUT / 'lead_hand_profiles.html')
print('CSV measurements, profiles, eligibility, duration review, sensitivity, outcome associations, and input manifest:',OUT)
display(FileLink(str(OUT / 'lead_hand_profiles.html')))